# 04 - Interactive Agent Authentication and User Authorization

**Learning objectives**
- Understand interactive agent authentication patterns
- Learn how users authenticate to agent blueprints
- Configure redirect URIs for OAuth authorization
- Construct authorization URLs for user consent
- Validate user tokens in agent APIs
- Implement on-behalf-of (OBO) flow for delegated API access

**Prerequisites**
- Completed previous notebooks (01, 02, 03)
- Understanding of OAuth 2.0 authorization code flow
- Agent blueprint configured with `a365.ps1`

**What you'll learn:**
- **User Authentication**: How users sign in and grant consent to agents
- **Token Validation**: Verifying user tokens in agent APIs
- **User Authorization**: Requesting delegated permissions from users
- **OBO Flow**: Exchanging user tokens for downstream API tokens
- **Redirect URIs**: Configuring callback endpoints for OAuth

**Documentation references:**
- [Authenticate users in interactive agents](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/interactive-agent-authenticate-user)
- [Configure user authorization](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/interactive-agent-request-user-authorization)

## Interactive Agent Architecture

Interactive agents act on behalf of human users. Here's how the flow works:

```
┌────────────┐                                    ┌─────────────┐
│   User     │                                    │  Azure      │
│            │──1. Sign in────────────────────────│  Entra ID   │
│            │◄─2. Authorization code─────────────│             │
└────────────┘                                    └─────────────┘
      │                                                  ▲
      │                                                  │
      ▼                                                  │
┌────────────┐      3. Code for token       ┌──────────┴──────┐
│  Client    │──────────────────────────────►│  Token          │
│  App       │◄─4. User access token─────────│  Endpoint       │
│            │                                └─────────────────┘
└────────────┘
      │
      │ 5. Call agent API with user token
      ▼
┌────────────┐      6. Validate token         ┌─────────────┐
│  Agent     │───────────────────────────────►│  Agent ID   │
│  Web API   │◄──7. Token claims──────────────│  SDK        │
│            │                                 │  /Validate  │
└────────────┘                                 └─────────────┘
      │
      │ 8. Exchange for downstream token (OBO)
      ▼
┌────────────┐      9. Get auth header         ┌─────────────┐
│  Agent     │───────────────────────────────►│  Agent ID   │
│  Web API   │◄──10. Downstream token─────────│  SDK        │
│            │                                 │  /AuthHeader│
└────────────┘                                 └─────────────┘
      │
      │ 11. Call Graph on behalf of user
      ▼
┌────────────┐
│ Microsoft  │
│ Graph API  │
└────────────┘
```

### Key Players

1. **User**: Human user who wants to use the agent
2. **Client App**: Frontend application (web, mobile, desktop)
3. **Azure Entra ID**: Identity provider for authentication and authorization
4. **Agent Web API**: Your agent service that processes requests
5. **Agent ID SDK**: Helper service for token operations
6. **Microsoft Graph**: Downstream API the agent calls on user's behalf

## Setup: Load Configuration

Load our agent blueprint configuration.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment
load_dotenv()

config = {
    "tenant_id": os.getenv("AZURE_TENANT_ID"),
    "client_id": os.getenv("AZURE_CLIENT_ID"),
    "credential_type": os.getenv("AZURE_CLIENT_CREDENTIAL_TYPE"),
    "blueprint_id": os.getenv("AGENT_BLUEPRINT_ID"),
    "identifier_uri": os.getenv("AGENT_BLUEPRINT_IDENTIFIER_URI"),
    "scope": os.getenv("AGENT_BLUEPRINT_SCOPE")
}

# Validate
missing = [k for k, v in config.items() if not v]
if missing:
    raise ValueError(f"❌ Missing: {', '.join(missing)}")

print("✅ Configuration loaded")
print(f"   Tenant: {config['tenant_id']}")
print(f"   Client: {config['client_id']}")
print(f"   Identifier URI: {config['identifier_uri']}")
print(f"   Scope: {config['scope']}")

## Step 1: Configure Redirect URI

For interactive authentication, the agent blueprint must have a **redirect URI** configured. This is where Azure Entra ID sends users after they grant consent.

### What is a redirect URI?

- **Purpose**: OAuth callback endpoint for authorization code
- **Format**: Must be HTTPS (except localhost for development)
- **Examples**:
  - `https://myagentapp.com/authorize`
  - `https://localhost:5001/callback` (dev)
  - `http://localhost:3000/auth/callback` (local dev)

### How to configure

**Option 1: Azure Portal**
1. Go to Entra ID → App registrations → your blueprint
2. Navigate to "Authentication"
3. Click "Add a platform" → "Web"
4. Enter your redirect URI
5. Save

**Option 2: Microsoft Graph API** (shown below)

Let's add a redirect URI programmatically:

In [ ]:
import msal
import requests

# Get Graph token
# Import shared credential function from utils.py
from utils import get_graph_token

# Note: get_graph_token(config) is now imported from utils.py
# It supports both client secret and certificate authentication.

# Add redirect URI to blueprint
def add_redirect_uri(redirect_uri):
    """
    Add a redirect URI to the agent blueprint.
    
    Args:
        redirect_uri: The URI to add (e.g., https://myapp.com/authorize)
    """
    token = get_graph_token(config)
    
    # Use beta endpoint for agent blueprints
    url = f"https://graph.microsoft.com/beta/applications/{config['blueprint_id']}"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    # First, get existing redirect URIs
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"Failed to get application: {response.text}")
    
    app_data = response.json()
    existing_uris = app_data.get('web', {}).get('redirectUris', [])
    
    # Add new URI if not already present
    if redirect_uri not in existing_uris:
        existing_uris.append(redirect_uri)
        
        # Update application
        update_data = {
            "web": {
                "redirectUris": existing_uris
            }
        }
        
        response = requests.patch(url, headers=headers, json=update_data)
        
        if response.status_code == 204:
            print(f"✅ Added redirect URI: {redirect_uri}")
            return True
        else:
            error = response.json()
            raise Exception(
                f"Failed to update application ({response.status_code}): "
                f"{error.get('error', {}).get('message', error)}"
            )
    else:
        print(f"ℹ️  Redirect URI already configured: {redirect_uri}")
        return False

# Configure redirect URI for local development
redirect_uri = "http://localhost:3000/auth/callback"

print("🔧 Configuring redirect URI...\n")

try:
    add_redirect_uri(redirect_uri)
    print(f"\n✅ Blueprint configured for interactive authentication")
    print(f"   Redirect URI: {redirect_uri}")
except Exception as e:
    print(f"❌ Failed to configure redirect URI: {e}")
    print("\n⚠️  You may need to configure this manually in Azure Portal:")
    print("   1. Go to Entra ID → App registrations → your blueprint")
    print("   2. Navigate to Authentication")
    print("   3. Add redirect URI: http://localhost:3000/auth/callback")

## Step 2: Construct Authorization URL

To authenticate users, the client app redirects them to an authorization URL. This URL initiates the OAuth 2.0 authorization code flow.

### Authorization URL Components

```
https://login.microsoftonline.com/{tenant}/oauth2/v2.0/authorize?
  client_id={agent-blueprint-client-id}
  &response_type=code
  &redirect_uri={encoded-redirect-uri}
  &response_mode=query
  &scope={blueprint-scope}
  &state={random-state}
```

### Key Parameters

- **`client_id`**: Agent blueprint's application ID
- **`response_type`**: Set to `code` for authorization code flow
- **`redirect_uri`**: Where to send user after authentication (must match configured URI)
- **`response_mode`**: `query` (code in URL) or `fragment` (code in hash)
- **`scope`**: Permissions requested (e.g., `api://{blueprintId}/access_agent`)
- **`state`**: Random value for CSRF protection (recommended)

### Full Scope Format

For agent blueprints, the scope follows this pattern:
```
api://{blueprint-client-id}/{scope-name}
```

From our configuration:
- Identifier URI: `{identifier_uri}` (e.g., `api://12345678-...`)
- Scope: `{scope}` (e.g., `access_agent`)
- **Full scope**: `{identifier_uri}/{scope}`

Let's construct the authorization URL:

In [ ]:
from urllib.parse import urlencode, quote
import secrets

def build_authorization_url(
    tenant_id,
    client_id,
    redirect_uri,
    scope,
    state=None,
    response_mode="query"
):
    """
    Build an OAuth 2.0 authorization URL for user authentication.
    
    Args:
        tenant_id: Azure AD tenant ID
        client_id: Agent blueprint client ID
        redirect_uri: Callback URI
        scope: Permission scope(s) to request
        state: Random state for CSRF protection (auto-generated if not provided)
        response_mode: 'query' or 'fragment'
    
    Returns:
        Authorization URL
    """
    if not state:
        state = secrets.token_urlsafe(32)  # Generate random state
    
    # Build authorization endpoint
    auth_endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/authorize"
    
    # Build query parameters
    params = {
        "client_id": client_id,
        "response_type": "code",
        "redirect_uri": redirect_uri,
        "response_mode": response_mode,
        "scope": scope,
        "state": state
    }
    
    # Construct URL
    auth_url = f"{auth_endpoint}?{urlencode(params)}"
    
    return auth_url, state

# Build authorization URL for our agent blueprint
print("🔗 Building authorization URL...\n")

# Construct full scope
full_scope = f"{config['identifier_uri']}/{config['scope']}"

auth_url, state = build_authorization_url(
    tenant_id=config['tenant_id'],
    client_id=config['client_id'],
    redirect_uri=redirect_uri,
    scope=full_scope
)

print("✅ Authorization URL constructed:\n")
print(f"URL: {auth_url}\n")
print(f"📋 URL Components:")
print(f"   Tenant: {config['tenant_id']}")
print(f"   Client ID: {config['client_id']}")
print(f"   Redirect URI: {redirect_uri}")
print(f"   Scope: {full_scope}")
print(f"   State: {state[:20]}... (CSRF token)")

print("\n🌐 User Flow:")
print("   1. Client app redirects user to this URL")
print("   2. User signs in to Microsoft")
print("   3. User grants consent to the requested scope")
print("   4. User is redirected back to: {redirect_uri}?code=...&state=...")
print("   5. Client app exchanges code for access token")

# Save for reference
auth_config = {
    "authorization_url": auth_url,
    "redirect_uri": redirect_uri,
    "scope": full_scope,
    "state": state
}

with open('auth-config.json', 'w') as f:
    json.dump(auth_config, f, indent=2)
print("\n💾 Configuration saved to: auth-config.json")

## Step 3: Token Exchange (Authorization Code → Access Token)

After the user grants consent, Azure redirects them back with an authorization code:

```
http://localhost:3000/auth/callback?code=ABC123...&state=xyz789...
```

The client app must:
1. **Validate state**: Ensure it matches the original state (CSRF protection)
2. **Exchange code for token**: POST to token endpoint

### Token Exchange Request

```http
POST https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token
Content-Type: application/x-www-form-urlencoded

client_id={client-id}
&grant_type=authorization_code
&code={authorization-code}
&redirect_uri={redirect-uri}
&scope={scope}
&client_secret={client-secret}  # Only for confidential clients
```

Let's implement the token exchange:

In [ ]:
def exchange_code_for_token(
    tenant_id,
    client_id,
    authorization_code,
    redirect_uri,
    scope,
    client_secret=None
):
    """
    Exchange authorization code for access token.
    
    Args:
        tenant_id: Azure AD tenant ID
        client_id: Agent blueprint client ID
        authorization_code: Code from callback
        redirect_uri: Same URI used in authorization request
        scope: Same scope used in authorization request
        client_secret: Client secret (for confidential clients)
    
    Returns:
        Token response with access_token, refresh_token, etc.
    """
    token_endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    
    data = {
        "client_id": client_id,
        "grant_type": "authorization_code",
        "code": authorization_code,
        "redirect_uri": redirect_uri,
        "scope": scope
    }
    
    # Add client secret for confidential clients
    if client_secret:
        data["client_secret"] = client_secret
    
    response = requests.post(
        token_endpoint,
        data=data,
        headers={"Content-Type": "application/x-www-form-urlencoded"}
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        error = response.json()
        raise Exception(
            f"Token exchange failed ({response.status_code}): "
            f"{error.get('error_description', error)}"
        )

# Simulate token exchange (in reality, this happens after user completes auth)
print("🔄 Token Exchange Process:\n")
print("📝 Steps:")
print("   1. User completes sign-in at authorization URL")
print("   2. Azure redirects to: {redirect_uri}?code=ABC...&state=xyz...")
print("   3. Client app validates state parameter")
print("   4. Client app exchanges authorization code for tokens")
print("   5. Client app receives:")
print("      • access_token: User's access token for agent blueprint")
print("      • refresh_token: For renewing expired tokens")
print("      • id_token: User identity information")
print("      • expires_in: Token lifetime (typically 3600 seconds)")

print("\n🔍 Example Token Response:\n")
example_response = {
    "token_type": "Bearer",
    "scope": full_scope,
    "expires_in": 3600,
    "access_token": "eyJ0eXAiOiJKV1Qi...",
    "refresh_token": "0.ARoA...",
    "id_token": "eyJ0eXAiOiJKV1Qi..."
}
print(json.dumps(example_response, indent=2))

print("\n⚠️  Important Security Notes:")
print("   • Always validate the 'state' parameter to prevent CSRF attacks")
print("   • Store refresh tokens securely (encrypted)")
print("   • Never log or expose access tokens")
print("   • Use HTTPS for all OAuth endpoints (except localhost dev)")

## Step 4: Validate User Token in Agent API

When the client app calls the agent API with a user token, the agent must validate it.

### Validation Steps

1. **Decode JWT**: Extract header and claims
2. **Verify signature**: Ensure token hasn't been tampered with
3. **Check audience**: Token must be for this agent blueprint
4. **Check issuer**: Must be from trusted Azure AD tenant
5. **Check expiration**: Token must not be expired
6. **Extract user info**: Get user claims for authorization

### Using Microsoft.Identity.Web (C#/.NET)

```csharp
// In ASP.NET Core API
builder.Services.AddAuthentication(JwtBearerDefaults.AuthenticationScheme)
    .AddMicrosoftIdentityWebApi(builder.Configuration.GetSection("AzureAd"));

[Authorize]  // Validates token automatically
[Route("api/agent")]
public class AgentController : ControllerBase
{
    [HttpGet("profile")]
    public IActionResult GetUserProfile()
    {
        var userName = User.Identity.Name;
        var userId = User.FindFirst("oid")?.Value;
        return Ok(new { userName, userId });
    }
}
```

### Using Python (PyJWT)

Let's implement token validation in Python:

In [ ]:
import jwt
from jwt import PyJWKClient
from datetime import datetime, timezone

def validate_user_token(access_token, expected_audience, expected_tenant_id):
    """
    Validate a user access token.
    
    Args:
        access_token: The JWT token to validate
        expected_audience: Expected audience (agent blueprint client ID)
        expected_tenant_id: Expected tenant ID
    
    Returns:
        Decoded token claims if valid
    
    Raises:
        Exception if validation fails
    """
    # Decode header to get key ID (kid)
    unverified_header = jwt.get_unverified_header(access_token)
    kid = unverified_header.get('kid')
    
    # Get signing keys from Microsoft's JWKS endpoint
    jwks_uri = f"https://login.microsoftonline.com/{expected_tenant_id}/discovery/v2.0/keys"
    jwks_client = PyJWKClient(jwks_uri)
    
    # Get the signing key
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)
    
    # Validate and decode token
    try:
        decoded = jwt.decode(
            access_token,
            signing_key.key,
            algorithms=["RS256"],
            audience=expected_audience,
            issuer=f"https://login.microsoftonline.com/{expected_tenant_id}/v2.0"
        )
        
        # Additional validation
        now = datetime.now(timezone.utc).timestamp()
        
        if decoded.get('exp', 0) < now:
            raise Exception("Token has expired")
        
        if decoded.get('nbf', 0) > now:
            raise Exception("Token not yet valid")
        
        return decoded
        
    except jwt.ExpiredSignatureError:
        raise Exception("Token has expired")
    except jwt.InvalidAudienceError:
        raise Exception(f"Invalid audience. Expected: {expected_audience}")
    except jwt.InvalidIssuerError:
        raise Exception(f"Invalid issuer. Expected tenant: {expected_tenant_id}")
    except Exception as e:
        raise Exception(f"Token validation failed: {str(e)}")

# Example: Validate a user token
print("🔐 Token Validation Process:\n")
print("📝 Validation Steps:")
print("   1. Extract JWT header and claims")
print("   2. Fetch signing keys from Microsoft's JWKS endpoint")
print("   3. Verify JWT signature using public key")
print("   4. Validate audience matches agent blueprint")
print("   5. Validate issuer is trusted Azure AD tenant")
print("   6. Check token expiration timestamps")
print("   7. Extract user claims for authorization")

print("\n🏷️  Key Claims to Extract:\n")
claims_info = {
    "oid": "User's object ID in Azure AD",
    "sub": "Subject identifier (unique per user per app)",
    "name": "User's display name",
    "preferred_username": "User's email or UPN",
    "email": "User's email address",
    "roles": "Application roles assigned to user",
    "scp": "Delegated permission scopes granted",
    "tid": "Tenant ID",
    "aud": "Audience (should be agent blueprint ID)",
    "exp": "Expiration timestamp",
    "iat": "Issued at timestamp"
}

for claim, description in claims_info.items():
    print(f"   • {claim}: {description}")

print("\n✅ After validation, agent can:")
print("   • Trust the user identity")
print("   • Make authorization decisions based on user claims")
print("   • Use the token to call downstream APIs on user's behalf")

## Step 5: On-Behalf-Of (OBO) Flow

After validating the user token, the agent can exchange it for a downstream API token using the **On-Behalf-Of (OBO)** flow.

### Why OBO?

- User authenticates to agent API
- Agent needs to call Microsoft Graph **as the user**
- Agent exchanges user token for Graph token
- Graph actions appear as if user performed them

### OBO Token Request

```http
POST https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token
Content-Type: application/x-www-form-urlencoded

client_id={agent-blueprint-client-id}
&grant_type=urn:ietf:params:oauth:grant-type:jwt-bearer
&assertion={user-access-token}
&scope=https://graph.microsoft.com/.default
&client_secret={client-secret}
&requested_token_use=on_behalf_of
```

Let's implement OBO token exchange:

In [ ]:
def acquire_obo_token(
    tenant_id,
    client_id,
    user_token,
    downstream_scope,
    client_secret=None,
    client_certificate=None
):
    """
    Acquire a token for downstream API using On-Behalf-Of flow.
    
    Args:
        tenant_id: Azure AD tenant ID
        client_id: Agent blueprint client ID
        user_token: User's access token for agent
        downstream_scope: Scope for downstream API (e.g., Microsoft Graph)
        client_secret: Agent blueprint client secret
        client_certificate: Alternative to secret (not implemented here)
    
    Returns:
        Token response with access_token for downstream API
    """
    token_endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    
    data = {
        "client_id": client_id,
        "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
        "assertion": user_token,
        "scope": downstream_scope,
        "requested_token_use": "on_behalf_of"
    }
    
    # Add client authentication
    if client_secret:
        data["client_secret"] = client_secret
    elif client_certificate:
        # Certificate authentication would require client assertion
        # Not implemented in this example
        raise NotImplementedError("Certificate auth for OBO not implemented")
    else:
        raise ValueError("Must provide either client_secret or client_certificate")
    
    response = requests.post(
        token_endpoint,
        data=data,
        headers={"Content-Type": "application/x-www-form-urlencoded"}
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        error = response.json()
        raise Exception(
            f"OBO token acquisition failed ({response.status_code}): "
            f"{error.get('error_description', error)}"
        )

# Demonstrate OBO flow
print("🔄 On-Behalf-Of (OBO) Flow:\n")
print("📊 Flow Diagram:\n")
print("   User Token (for Agent)")
print("         │")
print("         ▼")
print("   ┌──────────────────┐")
print("   │  Agent API       │")
print("   │  (validates)     │")
print("   └──────────────────┘")
print("         │")
print("         │ OBO Request")
print("         ▼")
print("   ┌──────────────────┐")
print("   │  Azure Entra ID  │")
print("   │  (token endpoint)│")
print("   └──────────────────┘")
print("         │")
print("         ▼")
print("   Graph Token (for User)")
print("         │")
print("         ▼")
print("   ┌──────────────────┐")
print("   │ Microsoft Graph  │")
print("   │ (on behalf of    │")
print("   │  user)           │")
print("   └──────────────────┘")

print("\n🔑 OBO Request Parameters:\n")
obo_params = {
    "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
    "client_id": config['client_id'],
    "assertion": "<user-access-token>",
    "scope": "https://graph.microsoft.com/.default",
    "client_secret": "<blueprint-secret>",
    "requested_token_use": "on_behalf_of"
}
print(json.dumps(obo_params, indent=2))

print("\n✅ After OBO exchange:")
print("   • Agent has Graph token with user context")
print("   • Graph sees requests as coming from the user")
print("   • User's permissions apply (not agent's)")
print("   • Audit logs show user as the actor")

print("\n🎯 Example Use Cases:")
print("   • Agent reads user's emails on their behalf")
print("   • Agent creates calendar events for user")
print("   • Agent uploads files to user's OneDrive")
print("   • Agent sends emails from user's mailbox")

## Complete Interactive Flow Example

Let's put it all together with a complete example showing the end-to-end flow.

In [ ]:
print("🎬 Complete Interactive Agent Flow\n")
print("="*60)
print("\n👤 Scenario: User asks chat bot to read their unread emails\n")
print("="*60)

print("\n🔵 Phase 1: User Authentication")
print("-" * 40)
print("   1. User opens chat app")
print("   2. App checks if user is authenticated")
print("   3. If not, app builds authorization URL:")
print(f"      {auth_url[:80]}...")
print("   4. App redirects user to sign-in page")
print("   5. User signs in with Microsoft account")
print("   6. User sees consent screen:")
print("      'Chat Bot wants to access your profile and email'")
print("   7. User clicks 'Accept'")
print("   8. Azure redirects back with authorization code")
print("   9. App exchanges code for user token")
print("   ✅ User is now authenticated")

print("\n🔵 Phase 2: User Makes Request")
print("-" * 40)
print("   1. User types: 'Show me my unread emails'")
print("   2. App sends request to agent API:")
print("      POST /api/agent/emails")
print("      Authorization: Bearer <user-token>")
print("      { 'query': 'unread emails' }")

print("\n🔵 Phase 3: Agent Validates User Token")
print("-" * 40)
print("   1. Agent receives request with user token")
print("   2. Agent validates token:")
print("      • Verify signature")
print("      • Check audience (agent blueprint)")
print("      • Check expiration")
print("   3. Agent extracts user info:")
print("      • User ID: 12345678-...")
print("      • Email: user@contoso.com")
print("      • Name: John Doe")
print("   ✅ Token validated, user identified")

print("\n🔵 Phase 4: Agent Acquires Graph Token (OBO)")
print("-" * 40)
print("   1. Agent needs to access user's emails in Graph")
print("   2. Agent exchanges user token for Graph token:")
print("      POST /oauth2/v2.0/token")
print("      grant_type=urn:ietf:params:oauth:grant-type:jwt-bearer")
print("      assertion=<user-token>")
print("      scope=https://graph.microsoft.com/.default")
print("   3. Azure validates:")
print("      • Agent blueprint credentials")
print("      • User token validity")
print("      • Requested permissions")
print("   4. Azure issues Graph token with user context")
print("   ✅ Agent has Graph token on behalf of user")

print("\n🔵 Phase 5: Agent Calls Microsoft Graph")
print("-" * 40)
print("   1. Agent calls Graph API:")
print("      GET /v1.0/me/messages?$filter=isRead eq false")
print("      Authorization: Bearer <graph-token-obo>")
print("   2. Graph validates token and sees user context")
print("   3. Graph returns user's unread emails")
print("   4. Agent processes and formats results")
print("   ✅ Email data retrieved on behalf of user")

print("\n🔵 Phase 6: Agent Returns Response")
print("-" * 40)
print("   1. Agent sends formatted response to app:")
print("      {")
print("        'emails': [")
print("          {")
print("            'from': 'boss@contoso.com',")
print("            'subject': 'Q1 Report Deadline',")
print("            'preview': 'The report is due tomorrow...'")
print("          },")
print("          ...")
print("        ]")
print("      }")
print("   2. App displays emails to user")
print("   ✅ User sees their unread emails")

print("\n" + "="*60)
print("✅ Complete flow executed successfully!")
print("="*60)

print("\n🎯 Key Points:")
print("   • User authenticates once, stays signed in")
print("   • Agent never sees user's password")
print("   • Graph actions appear as if user performed them")
print("   • User can revoke agent access anytime")
print("   • All actions are audited under user's identity")

## Security Best Practices

### 1. State Parameter (CSRF Protection)
- Always generate random state for authorization requests
- Validate state matches when receiving callback
- Prevents cross-site request forgery attacks

### 2. Token Storage
- **Access tokens**: Short-lived, can be in memory
- **Refresh tokens**: Long-lived, must be encrypted at rest
- Never log tokens or include in error messages
- Use secure storage (encrypted database, Azure Key Vault)

### 3. Token Validation
- Always validate JWT signature
- Check audience, issuer, expiration
- Use established libraries (Microsoft.Identity.Web, PyJWT)
- Never skip validation in production

### 4. HTTPS Requirements
- All redirect URIs must use HTTPS (except localhost)
- All token endpoints must use HTTPS
- All API endpoints must use HTTPS

### 5. Client Credentials
- **Development**: Client secrets acceptable
- **Production**: Use certificates or managed identities
- Rotate secrets/certificates regularly
- Never commit credentials to source control

### 6. Scope Principle of Least Privilege
- Request only the permissions you need
- Use granular scopes (Mail.Read vs Mail.ReadWrite)
- Document why each permission is needed

### 7. Token Lifetime Management
- Cache tokens to minimize requests
- Refresh before expiration (not after)
- Handle 401 errors gracefully
- Clear cached tokens on sign-out

## Summary

✅ **What you've learned:**

1. **Interactive Agent Architecture**: How users, apps, agents, and APIs interact
2. **Redirect URIs**: Configuring OAuth callback endpoints
3. **Authorization URLs**: Building user sign-in flows
4. **Token Exchange**: Converting authorization codes to access tokens
5. **Token Validation**: Verifying user tokens in agent APIs
6. **OBO Flow**: Exchanging user tokens for downstream API tokens
7. **Security Best Practices**: Protecting tokens and preventing attacks

**Key concepts:**
- Interactive agents require user consent and authentication
- OAuth 2.0 authorization code flow enables secure user sign-in
- Agents must validate user tokens before processing requests
- OBO flow allows agents to call APIs on behalf of users
- Security is critical: HTTPS, state validation, token encryption

**Complete agent capabilities:**
- ✅ **Autonomous agents**: Act independently (Notebook 02)
- ✅ **Agent users**: Agents with user-like capabilities (Notebook 03)
- ✅ **Interactive agents**: Act on behalf of human users (This notebook)

**What's next:**
- Deploy agent API with authentication
- Build client app with user sign-in
- Implement OBO flow for Graph access
- Set up monitoring and logging
- Add multi-agent scenarios

**Additional resources:**
- [Microsoft identity platform documentation](https://learn.microsoft.com/en-us/entra/identity-platform/)
- [OAuth 2.0 specification](https://oauth.net/2/)
- [Agent Identity documentation](https://learn.microsoft.com/en-us/entra/agent-id/)
- [Microsoft Graph API](https://learn.microsoft.com/en-us/graph/api/overview)